# Fine-Tuning NLLB-200 pour la Traduction Éwé-Anglais

**Modèle** : `facebook/nllb-200-distilled-600M`  
**Tâche** : `ewe_Latn` → `eng_Latn` (Éwé → Anglais)  
**Données** : `data/processed/translation/` — ~54 k paires train / 6.7 k val / 6.7 k test  
**Méthode** : LoRA (PEFT) — seul ~0.5 % des paramètres est entraîné

### Plan
1. Installation & imports  
2. Configuration des hyperparamètres  
3. Chargement du dataset  
4. **Évaluation baseline** (modèle brut, avant fine-tuning) → sauvegarde JSON  
5. Prétraitement / tokenisation  
6. Configuration LoRA  
7. Entraînement + validation  
8. Évaluation finale (test set)  
9. Comparaison baseline vs fine-tuné

## 0. Installation des dépendances

In [ ]:
# Exécuter une seule fois (redémarrer le kernel ensuite si nécessaire)
# !pip install transformers datasets evaluate peft accelerate sacrebleu sentencepiece protobuf

## 1. Imports

In [ ]:
import json
import os
from pathlib import Path

import torch
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from peft import LoraConfig, TaskType, get_peft_model, PeftModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")
print(f"PyTorch : {torch.__version__}")

## 2. Configuration

Tous les hyperparamètres sont centralisés ici pour faciliter les expériences.

In [ ]:
# ── Modèle & chemins ────────────────────────────────────────────────────────
MODEL_NAME   = "facebook/nllb-200-distilled-600M"
OUTPUT_DIR   = "./output/nllb-ewe-eng"
ADAPTER_DIR  = "./output/nllb-ewe-eng/adapter"
RESULTS_FILE = "./output/resultats_evaluation.json"

# ── Paires de langues (codes BCP-47 NLLB) ────────────────────────────────────
SRC_LANG = "ewe_Latn"   # Éwé
TGT_LANG = "eng_Latn"   # Anglais  (remplacer par "fra_Latn" pour le français)

# ── Données ──────────────────────────────────────────────────────────────────
DATA_DIR = Path("data/processed/translation")

# ── Tokenizer ────────────────────────────────────────────────────────────────
# max_length : longueur en TOKENS. Trop grand → explosion VRAM (attention O(n²)).
MAX_INPUT_LEN  = 128
MAX_TARGET_LEN = 128

# ── Entraînement ─────────────────────────────────────────────────────────────
# learning_rate : taille du pas de l'optimiseur.
#   Trop grand → la loss diverge ; trop petit → convergence lente.
#   Plus élevé qu'habituellement (3e-4 vs 5e-5) car seul LoRA est entraîné.
LEARNING_RATE = 3e-4

# per_device_train_batch_size : exemples par GPU par micro-étape.
#   Réduire si VRAM insuffisante ; compenser avec GRAD_ACCUM_STEPS.
BATCH_SIZE_TRAIN = 16
BATCH_SIZE_EVAL  = 32

# gradient_accumulation_steps : accumule N micro-gradients avant un update.
#   Batch effectif = BATCH_SIZE_TRAIN × GRAD_ACCUM_STEPS = 16 × 2 = 32.
GRAD_ACCUM_STEPS = 2

# num_train_epochs : tours complets sur le dataset d'entraînement.
NUM_EPOCHS = 3

# warmup_ratio : fraction des steps dédiée à la montée progressive du LR.
#   Évite un démarrage brutal qui endommagerait les poids pré-entraînés.
WARMUP_RATIO = 0.06

# weight_decay : pénalité L2 légère sur les poids — réduit l'overfitting.
WEIGHT_DECAY = 0.01

# ── LoRA ──────────────────────────────────────────────────────────────────────
# r (rang) : dimension des matrices d'adaptation A et B (ΔW = BA, rang r).
#   Plus r est grand → plus de capacité, mais plus coûteux. Plage : 4–64.
LORA_R = 16

# lora_alpha : facteur d'échelle appliqué à ΔW (mise à l'échelle par α/r ou α/√r).
#   Convention courante : alpha = 2 × r.
LORA_ALPHA = 32

# lora_dropout : régularisation sur les couches LoRA — utile sur petits corpus.
LORA_DROPOUT = 0.05

# target_modules : quelles projections linéaires adapter.
#   Pour NLLB (architecture M2M-100) : q_proj et v_proj sont les plus impactants.
LORA_TARGET_MODULES = ["q_proj", "v_proj"]

# ── Évaluation rapide ─────────────────────────────────────────────────────────
BASELINE_SAMPLE = 200   # exemples pour l'éval baseline (validation, rapide)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Configuration OK")
print(f"  Paire          : {SRC_LANG} → {TGT_LANG}")
print(f"  Batch effectif : {BATCH_SIZE_TRAIN} × {GRAD_ACCUM_STEPS} = {BATCH_SIZE_TRAIN * GRAD_ACCUM_STEPS}")

## 3. Chargement du dataset

Format de chaque ligne JSONL :
```json
{"translation": {"ewe_Latn": "...", "eng_Latn": "..."}}
```
Certaines lignes ont `fra_Latn` au lieu de `eng_Latn` — on les filtre.

In [ ]:
raw = load_dataset(
    "json",
    data_files={
        "train":      str(DATA_DIR / "train.jsonl"),
        "validation": str(DATA_DIR / "validation.jsonl"),
        "test":       str(DATA_DIR / "test.jsonl"),
    },
)

# Garder uniquement les paires qui ont les deux langues cibles
def has_pair(example):
    t = example["translation"]
    return (
        SRC_LANG in t and TGT_LANG in t
        and bool(t[SRC_LANG]) and bool(t[TGT_LANG])
    )

raw = raw.filter(has_pair)

print(raw)
print(f"\nExemple train[0] :")
print(json.dumps(raw["train"][0], ensure_ascii=False, indent=2))

## 3.b Nettoyage du bruit

On retire ici deux bruits simples mais nuisibles :
- les exemples où la source et la cible sont identiques (copie au lieu de traduction)
- les cibles anglaises qui contiennent des caractères typiques de l'éwé

In [ ]:
import re

# Caractères typiques de l'éwé qu'on ne veut pas retrouver dans une cible anglaise.
EWE_CHARS = set("ŋɖɔɛʋƒãẽĩõũ")

# Références bibliques seules du type 15:1-33.
BIBLE_REF_RE = re.compile(r"^\s*\d{1,3}:\d{1,3}(?:-\d{1,3})?\s*$")

def is_same_source_target(example, src_lang=SRC_LANG, tgt_lang=TGT_LANG):
    src = example["translation"].get(src_lang, "").strip()
    tgt = example["translation"].get(tgt_lang, "").strip()
    return bool(src) and bool(tgt) and src == tgt

def is_bible_reference_only(example, src_lang=SRC_LANG, tgt_lang=TGT_LANG):
    src = example["translation"].get(src_lang, "").strip()
    tgt = example["translation"].get(tgt_lang, "").strip()
    return bool(BIBLE_REF_RE.match(src)) and src == tgt

def has_ewe_chars_in_english_target(example, tgt_lang=TGT_LANG, min_count=2):
    if tgt_lang != "eng_Latn":
        return False
    tgt = example["translation"].get(tgt_lang, "")
    count = sum(ch in EWE_CHARS for ch in tgt)
    return count >= min_count

def is_noisy(example, src_lang=SRC_LANG, tgt_lang=TGT_LANG):
    return (
        is_same_source_target(example, src_lang, tgt_lang)
        or is_bible_reference_only(example, src_lang, tgt_lang)
        or has_ewe_chars_in_english_target(example, tgt_lang)
    )

raw_before_cleaning = raw
raw = raw.filter(lambda ex: not is_noisy(ex, SRC_LANG, TGT_LANG))

print("Nettoyage terminé.")
for split in ["train", "validation", "test"]:
    removed = len(raw_before_cleaning[split]) - len(raw[split])
    print(f"  {split:<10} : supprimés={removed} | restants={len(raw[split])}")
    

## 4. Évaluation Baseline

On mesure les performances du modèle **sans aucun fine-tuning** sur notre dataset.  
Ce score de référence permettra de quantifier l'apport de l'entraînement.

**Métriques** :
- **BLEU** : recouvrement de n-grammes de mots (0–100, ↑ = mieux)
- **chrF++** : n-grammes de caractères + bigrammes de mots — plus robuste pour les langues à morphologie riche comme l'éwé

In [ ]:
sacrebleu_metric = evaluate.load("sacrebleu")
chrf_metric      = evaluate.load("chrf")

print("Chargement du modèle baseline…")
tokenizer_base = AutoTokenizer.from_pretrained(MODEL_NAME)
model_base = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
model_base.eval()
print("Modèle chargé.")

In [ ]:
def translate_batch(model, tokenizer, sources, src_lang, tgt_lang,
                    batch_size=16, max_new_tokens=128):
    """Génère les traductions pour une liste de phrases source."""
    tokenizer.src_lang = src_lang
    forced_bos = tokenizer.convert_tokens_to_ids(tgt_lang)
    all_preds = []

    for i in range(0, len(sources), batch_size):
        batch = sources[i : i + batch_size]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_LEN,
        ).to(device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                forced_bos_token_id=forced_bos,
                max_new_tokens=max_new_tokens,
                num_beams=4,
            )
        decoded = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
        all_preds.extend(decoded)

    return all_preds


def compute_metrics_on_split(model, tokenizer, dataset_split, n_samples=None):
    """Calcule BLEU et chrF++ sur un split (ou n_samples exemples)."""
    if n_samples:
        dataset_split = dataset_split.select(range(min(n_samples, len(dataset_split))))

    # Filtrer les paires où source ou référence est None/vide
    pairs = [
        (ex["translation"].get(SRC_LANG), ex["translation"].get(TGT_LANG))
        for ex in dataset_split
        if ex["translation"].get(SRC_LANG) and ex["translation"].get(TGT_LANG)
    ]
    sources    = [p[0] for p in pairs]
    references = [p[1] for p in pairs]

    print(f"  Génération de {len(sources)} traductions…")
    predictions = translate_batch(model, tokenizer, sources, SRC_LANG, TGT_LANG)

    bleu = sacrebleu_metric.compute(
        predictions=predictions, references=[[r] for r in references]
    )
    chrf = chrf_metric.compute(
        predictions=predictions, references=[[r] for r in references], word_order=2
    )

    return {
        "bleu":   round(bleu["score"], 2),
        "chrf++": round(chrf["score"], 2),
        "n":      len(sources),
    }


In [ ]:
print("=== BASELINE — validation rapide ===")
baseline_val = compute_metrics_on_split(
    model_base, tokenizer_base, raw["validation"], n_samples=BASELINE_SAMPLE
)
print(f"  BLEU   : {baseline_val['bleu']}")
print(f"  chrF++ : {baseline_val['chrf++']}")

In [ ]:
print("=== BASELINE — test complet ===")
baseline_test = compute_metrics_on_split(model_base, tokenizer_base, raw["test"])
print(f"  BLEU   : {baseline_test['bleu']}")
print(f"  chrF++ : {baseline_test['chrf++']}")

# ── Sauvegarde des résultats baseline ─────────────────────────────────────────
results = {
    "modele": MODEL_NAME,
    "paire":  f"{SRC_LANG} → {TGT_LANG}",
    "baseline": {
        "validation_sample": baseline_val,
        "test":              baseline_test,
    },
    "fine_tune": {},
}

with open(RESULTS_FILE, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"\nBaseline sauvegardé dans {RESULTS_FILE}")

# Libérer la VRAM avant l'entraînement
del model_base, tokenizer_base
if device == "cuda":
    torch.cuda.empty_cache()

## 5. Prétraitement / Tokenisation

Le tokenizer transforme chaque phrase en `input_ids` (liste d'entiers).  
- `max_length` tronque les phrases trop longues  
- Le padding est géré dynamiquement par `DataCollatorForSeq2Seq` (plus efficace que de pré-padder)  
- Les tokens de padding dans les **labels** sont remplacés par **-100** → ignorés par la cross-entropy

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.src_lang = SRC_LANG
FORCED_BOS_TOKEN_ID = tokenizer.convert_tokens_to_ids(TGT_LANG)


def preprocess(batch):
    sources = [ex[SRC_LANG] for ex in batch["translation"]]
    targets = [ex[TGT_LANG] for ex in batch["translation"]]

    model_inputs = tokenizer(
        sources,
        text_target=targets,
        max_length=MAX_INPUT_LEN,
        max_target_length=MAX_TARGET_LEN,
        truncation=True,
    )

    return model_inputs


tokenized = raw.map(
    preprocess,
    batched=True,
    remove_columns=raw["train"].column_names,
    desc="Tokenisation",
)
print(tokenized)


In [ ]:
# DataCollatorForSeq2Seq : pad chaque batch à la longueur max DU BATCH
# (évite de pré-padder à une longueur fixe globale, moins de calcul inutile)
# pad_to_multiple_of=8 : optimise les Tensor Cores sur GPU Ampere+
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=None,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)

## 6. Configuration LoRA

**LoRA** (Low-Rank Adaptation) gèle tous les poids originaux et ajoute deux petites matrices
$A \in \mathbb{R}^{r \times d_{in}}$ et $B \in \mathbb{R}^{d_{out} \times r}$ telles que :

$$\Delta W = BA, \quad \text{rang}(\Delta W) = r \ll d$$

Seuls $A$ et $B$ sont entraînés → environ **0.5 % des paramètres** du modèle total.

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

peft_config = LoraConfig(
    task_type       = TaskType.SEQ_2_SEQ_LM,
    r               = LORA_R,                # rang : capacité d'adaptation
    lora_alpha      = LORA_ALPHA,             # facteur d'échelle de ΔW
    lora_dropout    = LORA_DROPOUT,           # régularisation
    target_modules  = LORA_TARGET_MODULES,    # projections linéaires à adapter
    bias            = "none",
    use_rslora      = True,   # rsLoRA : échelle par α/√r (plus stable pour r élevé)
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
# Attendu : trainable params ≈ 3.5 M / 614 M (≈ 0.57 %)

## 7. Entraînement

`Seq2SeqTrainer` gère la boucle d'entraînement, les checkpoints, la mixed precision et le suivi des métriques de traduction.

In [ ]:
def compute_metrics(eval_preds):
    """Calcule BLEU et chrF++ à chaque eval_step pendant l'entraînement."""
    pred_ids, label_ids = eval_preds

    # Remplacer les -100 (padding labels) par le token pad du tokenizer
    label_ids = np.where(label_ids != -100, label_ids, tokenizer.pad_token_id)

    predictions = tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    references  = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    bleu = sacrebleu_metric.compute(
        predictions=predictions, references=[[r] for r in references]
    )
    chrf = chrf_metric.compute(
        predictions=predictions, references=[[r] for r in references], word_order=2
    )

    return {
        "bleu":   round(bleu["score"], 2),
        "chrf++": round(chrf["score"], 2),
    }

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir = OUTPUT_DIR,

    # ── Durée ────────────────────────────────────────────────────────────────
    num_train_epochs = NUM_EPOCHS,
    eval_steps       = 500,   # évaluer toutes les 500 étapes
    save_steps       = 500,   # sauvegarder un checkpoint toutes les 500 étapes
    logging_steps    = 100,   # afficher les logs toutes les 100 étapes

    # ── Batch & mémoire ──────────────────────────────────────────────────────
    per_device_train_batch_size = BATCH_SIZE_TRAIN,
    per_device_eval_batch_size  = BATCH_SIZE_EVAL,
    gradient_accumulation_steps = GRAD_ACCUM_STEPS,
    # gradient_checkpointing : recalcule les activations au lieu de les stocker.
    # Économise ~60 % de VRAM au prix de ~20 % de ralentissement.
    gradient_checkpointing = True,

    # ── Learning rate & scheduler ────────────────────────────────────────────
    learning_rate     = LEARNING_RATE,
    warmup_ratio      = WARMUP_RATIO,
    # cosine : montée linéaire pendant le warmup, puis décroissance en cosinus.
    # Meilleur que constant ou linéaire pour le fine-tuning.
    lr_scheduler_type = "cosine",
    weight_decay      = WEIGHT_DECAY,

    # ── Précision mixte ──────────────────────────────────────────────────────
    # fp16 : calcul en 16 bits → 2× plus rapide et 2× moins de VRAM.
    # Sur GPU Ampere+ (RTX 30xx, A100), remplacer par bf16=True (plus stable).
    fp16 = (device == "cuda"),

    # ── Génération pendant l'évaluation ─────────────────────────────────────
    # predict_with_generate=True : utilise model.generate() pour les métriques
    # (obligatoire pour BLEU/chrF++ qui comparent des phrases entières).
    predict_with_generate  = True,
    generation_max_length  = MAX_TARGET_LEN,

    # ── Stratégie de sauvegarde ──────────────────────────────────────────────
    eval_strategy          = "steps",   # renommé depuis evaluation_strategy
    save_strategy          = "steps",
    load_best_model_at_end = True,          # recharge automatiquement le meilleur ckpt
    metric_for_best_model  = "chrf++",      # chrF++ plus robuste que BLEU pour l'éwé
    greater_is_better      = True,
    save_total_limit       = 2,             # conserver seulement 2 checkpoints

    report_to = "none",   # désactiver W&B/TensorBoard (mettre "wandb" si besoin)
)

trainer = Seq2SeqTrainer(
    model              = model,
    args               = training_args,
    train_dataset      = tokenized["train"],
    eval_dataset       = tokenized["validation"],
    processing_class   = tokenizer,   # renommé depuis 'tokenizer' dans transformers ≥ 4.46
    data_collator      = data_collator,
    compute_metrics    = compute_metrics,
)

print("Trainer configuré. Démarrage de l'entraînement…")


In [ ]:
train_result = trainer.train()

# Sauvegarder uniquement l'adaptateur LoRA (~30 Mo, pas le modèle entier ~1.2 Go)
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print(f"\nAdaptateur LoRA sauvegardé dans : {ADAPTER_DIR}")
print(f"Étapes   : {train_result.global_step}")
print(f"Loss train finale : {train_result.training_loss:.4f}")

## 8. Évaluation Finale sur le Test Set

On fusionne l'adaptateur LoRA avec le modèle de base (`merge_and_unload`) pour obtenir un modèle standard — plus rapide à l'inférence.

In [ ]:
print("Chargement du meilleur modèle fine-tuné…")
base_model_ft = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model_ft      = PeftModel.from_pretrained(base_model_ft, ADAPTER_DIR)
model_ft      = model_ft.merge_and_unload()   # fusionne LoRA → modèle standard
model_ft      = model_ft.to(device)
model_ft.eval()

tokenizer_ft = AutoTokenizer.from_pretrained(ADAPTER_DIR)

print("=== FINE-TUNÉ — test complet ===")
ft_test = compute_metrics_on_split(model_ft, tokenizer_ft, raw["test"])
print(f"  BLEU   : {ft_test['bleu']}")
print(f"  chrF++ : {ft_test['chrf++']}")

## 9. Comparaison Baseline vs Fine-Tuné

In [ ]:
# Mise à jour du fichier de résultats
with open(RESULTS_FILE, "r", encoding="utf-8") as f:
    results = json.load(f)

results["fine_tune"]["test"] = ft_test

with open(RESULTS_FILE, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

# ── Tableau comparatif ────────────────────────────────────────────────────────
b = results["baseline"]["test"]
f = results["fine_tune"]["test"]

delta_bleu = round(f["bleu"]   - b["bleu"],   2)
delta_chrf = round(f["chrf++"] - b["chrf++"], 2)

print(f"{'Métrique':<12} {'Baseline':>10} {'Fine-tuné':>10} {'Δ':>8}")
print("-" * 44)
print(f"{'BLEU':<12} {b['bleu']:>10} {f['bleu']:>10} {delta_bleu:>+8}")
print(f"{'chrF++':<12} {b['chrf++']:>10} {f['chrf++']:>10} {delta_chrf:>+8}")
print(f"\nRésultats complets : {RESULTS_FILE}")

In [ ]:
# ── Quelques exemples de traduction ──────────────────────────────────────────
print("=== Exemples de traductions (test set) ===")
sample   = raw["test"].select(range(5))
sources  = [ex["translation"][SRC_LANG] for ex in sample]
refs     = [ex["translation"][TGT_LANG] for ex in sample]
preds    = translate_batch(model_ft, tokenizer_ft, sources, SRC_LANG, TGT_LANG)

for src, ref, pred in zip(sources, refs, preds):
    print(f"\nSource    : {src}")
    print(f"Référence : {ref}")
    print(f"Prédit    : {pred}")